#### Marketing Campaign Response Prediction - preprocessing, modelling, evaluation.
Trains Logistic Regression, Random Forest and XGBoost; compares them on
Accuracy / Precision / Recall / F1 / ROC-AUC; reports feature importance.

In [ ]:
import numpy as np, pandas as pd, json, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

In [ ]:
plt.rcParams.update({"figure.dpi":120,"axes.spines.top":False,"axes.spines.right":False,
                     "font.size":10,"axes.titleweight":"bold"})

In [ ]:
# ---------- LOAD ----------
df = pd.read_csv("C:/Users/CharlesOfosu/Downloads/Personal Project/Portfolio_Projects/03_marketing_campaign_prediction/data/marketing_campaign.csv")
TARGET = "responded"
print(f"Rows: {len(df)} | Response rate: {df[TARGET].mean()*100:.1f}%")

In [ ]:
# ---------- FEATURE ENGINEERING ----------
df["channel_match"] = (df["contact_channel"] == df["channel_preference"]).astype(int)
df["engagement_score"] = (df["email_open_rate"]*0.4 + df["email_click_rate"]*0.6)
df["purchases_per_year_tenure"] = df["num_purchases_last_year"] / (df["tenure_months"]/12 + 1)

In [ ]:
drop = ["customer_id", TARGET]
X = df.drop(columns=drop)
y = df[TARGET]

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()
print("Numeric features:", len(num_cols), "| Categorical:", len(cat_cols))

In [ ]:
# ---------- SPLIT ----------
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

In [ ]:
# Preprocessors: scaled for linear model, passthrough-numeric for trees
pre_scaled = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
pre_tree = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

In [ ]:
# ---------- MODELS ----------
models = {}
models["Logistic Regression"] = Pipeline([("pre", pre_scaled),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
models["Random Forest"] = Pipeline([("pre", pre_tree),
    ("clf", RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=5,
                                   class_weight="balanced", random_state=42, n_jobs=-1))])
try:
    from xgboost import XGBClassifier
    spw = (y_tr==0).sum()/(y_tr==1).sum()
    models["XGBoost"] = Pipeline([("pre", pre_tree),
        ("clf", XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                              subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
                              eval_metric="logloss", random_state=42, n_jobs=-1))])
    XGB_OK = True
except Exception as e:
    from sklearn.ensemble import HistGradientBoostingClassifier
    models["Gradient Boosting"] = Pipeline([("pre", pre_tree),
        ("clf", HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                                max_depth=4, random_state=42))])
    XGB_OK = False

In [ ]:
# ---------- TRAIN + EVALUATE ----------
cv = StratifiedKFold(5, shuffle=True, random_state=42)
results = {}
roc_data = {}
for name, pipe in models.items():
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)[:,1]
    auc_cv = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
    results[name] = {
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
        "cv_roc_auc": auc_cv,
    }
    fpr, tpr, _ = roc_curve(y_te, proba)
    roc_data[name] = (fpr, tpr, results[name]["roc_auc"])

In [ ]:
res_df = pd.DataFrame(results).T.round(3)
print("\n=== MODEL COMPARISON (held-out test set) ===")
print(res_df.to_string())
best = res_df["roc_auc"].idxmax()
print(f"\nBest model by ROC-AUC: {best} ({res_df.loc[best,'roc_auc']:.3f})")

In [ ]:
# Best model details
best_pipe = models[best]
best_pred = best_pipe.predict(X_te)
print(f"\n=== {best} classification report ===")
print(classification_report(y_te, best_pred, digits=3))
cm = confusion_matrix(y_te, best_pred)
print("Confusion matrix [ [TN FP] [FN TP] ]:\n", cm)

In [ ]:
# ---------- CHARTS ----------
# ROC curves
fig, ax = plt.subplots(figsize=(7,6))
for name,(fpr,tpr,auc) in roc_data.items():
    ax.plot(fpr,tpr,label=f"{name} (AUC={auc:.3f})", lw=2)
ax.plot([0,1],[0,1],"k--",alpha=.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves — model comparison"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("01_roc_curves.png"); 
plt.show()

In [ ]:
# Model comparison bar
fig, ax = plt.subplots(figsize=(9,5))
res_df[["precision","recall","f1","roc_auc"]].plot.bar(ax=ax)
ax.set_title("Model metrics comparison"); ax.set_ylabel("Score"); ax.set_ylim(0,1)
ax.set_xticklabels(res_df.index, rotation=0); ax.legend(loc="lower right", ncol=4)
plt.tight_layout(); plt.savefig("02_metric_comparison.png"); 
plt.show()

In [ ]:
# Confusion matrix (best)
fig, ax = plt.subplots(figsize=(5,4.5))
im=ax.imshow(cm, cmap="Blues")
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha="center",va="center",
        color="white" if v>cm.max()/2 else "black", fontsize=13)
ax.set_xticks([0,1]); ax.set_xticklabels(["No","Yes"]); ax.set_yticks([0,1]); ax.set_yticklabels(["No","Yes"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion matrix — {best}")
plt.tight_layout(); plt.savefig("03_confusion_matrix.png"); 
plt.show()

In [ ]:
# ---------- FEATURE IMPORTANCE ----------
# Use Random Forest importances mapped to feature names
rf = models["Random Forest"]
ohe = rf.named_steps["pre"].named_transformers_["cat"].named_steps["oh"]
feat_names = num_cols + list(ohe.get_feature_names_out(cat_cols))
imp = rf.named_steps["clf"].feature_importances_
fi = pd.Series(imp, index=feat_names).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8,6))
fi.sort_values().plot.barh(ax=ax, color="#3B6FB6")
ax.set_title("Top 15 feature importances (Random Forest)"); ax.set_xlabel("Importance")
plt.tight_layout(); plt.savefig("04_feature_importance.png"); 
plt.show()
print("\n=== TOP 10 FEATURES (Random Forest) ===")
print(fi.head(10).round(4).to_string())